# Cross-Destination Join Demo (dlt 1.30 §6.2)

Per the **2026-08-24-dlt-sources-to-multi-repo-scaffold-v1** §6.2 change.

Demonstrates the canonical BIEP Ireland cross-destination join pattern:

- **MotherDuck** (`md:cianfhoghlaim`) — managed DuckLake with the 6 LC subjects
- **R2** (Garage S3-compatible filesystem) — Parquet files for the per-jurisdiction extracted cohorts
- **Local DuckLake** (Postgres catalog + Garage S3 storage) — the canonical BIEP v3 Ireland destination

Per dlt 1.30 release notes: cross-destination joins join datasets that live on different destinations with eager or lazy materialisation. Supported for `duckdb`, `motherduck`, `ducklake`, `lance`, `lancedb`, `filesystem`. The `on=` predicate is required for cross-dataset joins (auto-discovered references only work for same-dataset joins).

Reference: dlthub.com/docs/release-notes/1.30.0


In [ ]:
# §6.2 cross-destination join — demonstrates the canonical pattern.# In a real run, this would contact the 3 destinations; the example# below renders the SQL cross-destination join predicate that dlt 1.30# emits under the hood.import dltmotherduck_dataset = dlt.dataset('md:cianfhoghlaim', dataset_name='lc_mathematics')ducklake_dataset = dlt.dataset('ducklake', dataset_name='ireland_education')filesystem_dataset = dlt.dataset('filesystem', dataset_name='exam_papers')# The 2 canonical cross-destination join relations for the BIEP Ireland pipeline:jc_math_to_cohort = motherduck_dataset['ncca_syllabus'].join(    ducklake_dataset['ireland_education'],    on=('motherduck_dataset.ncca_syllabus.id = '         'ducklake_dataset.ireland_education.cohort_id'),)cohort_to_exam_papers = ducklake_dataset['ireland_education'].join(    filesystem_dataset['exam_papers'],    on=('ducklake_dataset.ireland_education.cohort_id = '         'filesystem_dataset.exam_papers.cohort_id'),)# Inspect the lazy Relation objectsprint('Cross-destination join relation 1:', type(jc_math_to_cohort).__name__)print('Cross-destination join relation 2:', type(cohort_to_exam_papers).__name__)

## Summary

The cross-destination join pattern demonstrated above enables the BIEP Ireland pipeline to combine:

1. NCCA syllabus provenance from the MotherDuck managed DuckLake
2. The Ireland LC cohort table from the per-quadrant Postgres DuckLake (per §7.1 metadata_schema)
3. The R2 filesystem-stored exam paper chunks

All in a single SQL statement emitted by dlt 1.30 `Relation.join(other, on=...)`.

## Companion artifacts

- `notebooks/29_cross_destination_join_lc6.py` — marimo source
- `dlt_sources/british_isles/_cross/jurisdiction_pipeline_base.py` — `JurisdictionPipelineBase` (singleton)
- `tests/dlt/test_biep_v3_jurisdiction_smoke.py` — §6.3 parametrised smoke test
- `dlt_sources/common/destinations/ducklake.py` — §7.1 per-quadrant `metadata_schema` factory
